# RadioSkySims — Mapas do Céu

Fluxo de exemplo do bounded context **SkyMap** (`radioskysims-skymap`):
geração de mapas HEALPix (sintético offline + PySM3), operações
(suavização/máscara), análise (estatística + espectro angular Cℓ) e I/O (FITS).

A visualização é Mollweide via HEALPy, com figuras salvas em `plots/`.


## 1. Parâmetros de referência


In [ ]:
# ── Parâmetros de referência para mapas do céu ──
from pathlib import Path

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from radioskysims_viz import save_figure

# Resolução HEALPix (potência de 2) e frequência de referência.
NSIDE = 64            # → 49152 pixels
FREQ_GHZ = 1.0        # 1 GHz (banda L) — usado nos metadados/PySM
FRAME = "galactic"    # galactic | equatorial | ecliptic
SEED = 42

DATA_DIR = Path("data")
SKYMAP_DIR = DATA_DIR / "skymap"
SKYMAP_DIR.mkdir(parents=True, exist_ok=True)
print(f"NSide={NSIDE} → {hp.nside2npix(NSIDE)} pixels | freq={FREQ_GHZ} GHz | frame={FRAME}")


## 2. Mapa sintético (offline) — geração e visualização

Campo aleatório com espectro tipo-físico (`Cℓ ∝ ℓ^{-3/2}`) via `hp.synfast` — determinístico (seed) e sem rede.


In [ ]:
# Mapa sintético OFFLINE: campo gaussiano com espectro de potência dado.
from radioskysims_skymap.domain.skymap.factory import SkyMapFactory
from radioskysims_skymap.infrastructure.skymap.visualization import SkyMapVisualizer

rng = np.random.default_rng(SEED)
ell = np.arange(3, 200)
cls_input = 1.0 * ell**-1.5          # ~ Cℓ ∝ ℓ^{-3/2}
map_data = hp.synfast(cls_input, nside=NSIDE, lmax=199)

factory = SkyMapFactory(unit="uK_CMB", frame=FRAME, field="I")
skymap = factory.create(dataset=map_data, nside=NSIDE, freq_ghz=FREQ_GHZ)
skymap  # repr: nside/npix/unit/frame/field

# Mollweide (hp.mollview) → figura capturada e salva
vis = SkyMapVisualizer()
vis.plot_map(skymap, show=False, coord="G")
fig = plt.gcf()
fig
save_figure(fig, "skymap_analytic")


## 3. Geração com PySM3 (componentes físicos)

PySM3 monta mapas por componentes (synchrotron, dust, free-free, AME, CMB).
**Requer download de templates na 1ª execução** — se indisponível, apenas avisa.


In [ ]:
# PySM3: componentes físicos (synchrotron + dust) — opcional/online.
try:
    from radioskysims_skymap.infrastructure.skymap.providers.pysm import (
        PySMSkyMapGeneratorAdapter,
    )

    gen = PySMSkyMapGeneratorAdapter(unit="uK_CMB", field="I")
    pysm_map = gen.generate(
        nside=NSIDE, freq_ghz=FREQ_GHZ, components=["synchrotron", "dust"]
    )
    pysm_map
    vis.plot_map(pysm_map, show=False, coord="G")
    fig = plt.gcf()
    fig
    save_figure(fig, "skymap_pysm")
except Exception as exc:
    print("[aviso] PySM3 indisponível (templates?) —",
          type(exc).__name__, str(exc)[:200])


## 4. Operações: suavização e máscara

`SkyMapOperations.smooth` (gaussiana) e `SkyMask.apply` (máscara binária) — domínio puro, sem I/O.


In [ ]:
# Suavização gaussiana (FWHM em graus).
from radioskysims_skymap.domain.skymap.operations import SkyMapOperations

FWHM_DEG = 5.0
smoothed_data = SkyMapOperations.smooth(
    skymap.data, fwhm_rad=np.radians(FWHM_DEG)
)
smoothed = factory.create(dataset=smoothed_data, nside=NSIDE, freq_ghz=FREQ_GHZ)
smoothed

vis.plot_map(smoothed, show=False, coord="G")
fig = plt.gcf()
fig
save_figure(fig, "skymap_smoothed")


In [ ]:
# Máscara binária (metade do céu) — SkyMask.apply.
from radioskysims_skymap.domain.skymap.mask import SkyMask

mask = np.ones_like(skymap.data)
mask[len(mask) // 2:] = 0.0
masked_data = SkyMask.apply(skymap.data, mask)
masked = factory.create(dataset=masked_data, nside=NSIDE, freq_ghz=FREQ_GHZ)
masked

vis.plot_map(masked, show=False, coord="G")
fig = plt.gcf()
fig
save_figure(fig, "skymap_masked")


## 5. Análise: estatística e espectro angular

`SkyMapAnalysis` (média/RMS/min-max/Cℓ) + `SkyMapVisualizer.plot_cls`.


In [ ]:
# Estatísticas básicas + espectro angular Cℓ.
from radioskysims_skymap.domain.skymap.analysis import SkyMapAnalysis

lo, hi = SkyMapAnalysis.min_max(skymap.data)
print("mean:", round(SkyMapAnalysis.mean(skymap.data), 3), "uK_CMB")
print("rms :", round(SkyMapAnalysis.rms(skymap.data), 3), "uK_CMB")
print("min :", round(lo, 3), "| max:", round(hi, 3), "uK_CMB")

cls = SkyMapAnalysis.angular_power_spectrum(skymap.data, lmax=200)
print("Cℓ (ℓ=0..4):", [round(float(c), 3) for c in cls[:5]])

vis.plot_cls(cls, lmax=200, show=False)
fig = plt.gcf()
fig
save_figure(fig, "skymap_cls")


## 6. I/O: FITS

`HealpySkyMapProvider` grava e relê mapas em FITS.


In [ ]:
# Salva em FITS e relê (HealpySkyMapProvider).
from radioskysims_skymap.infrastructure.skymap.providers.healpy import (
    HealpySkyMapProvider,
)

provider = HealpySkyMapProvider()
path = SKYMAP_DIR / "skymap_analytic.fits"
provider.write_map(skymap, path, overwrite=True)
print("salvo:", path)

reloaded = provider.read_map(path, unit="uK_CMB", frame=FRAME)
reloaded
print("igual ao original:", np.allclose(reloaded.data, skymap.data))
